# Traversal Study: Scan-Order Geometry in Signal-Space AR Generation

## Research Question

Does **traversal geometry** measurably affect autoregressive likelihood
in serialized image generation, independent of model size and training budget?

We compare four scan orders at **equal token count (N = 1024)** and equal
architecture, varying only the pixel-visitation sequence.
The primary comparator is `raster_1spp` (plain raster, same N = 1024).
`raster_v5` (2 spp, flyback, N = 2208) is kept as a historical reference
from the V5 paper but is **not** a fair baseline here because N differs.

The locality metric quantifies mean sequence distance between spatially
adjacent pixel pairs. The paper tests whether that metric *predicts* bpd.

## Run Flags

| Flag | Content | Est. T4 time |
|------|---------|-------------|
| `RUN_LOCALITY` | Locality table — always fast | < 5 s |
| `RUN_HILBERT` | Hilbert, 3 seeds × 30 epochs | ≈ 3 h |
| `RUN_ALL` | All 4 traversals × 3 seeds × 30 epochs | ≈ 12 h |

**Recommended Kaggle sessions:**
- Session 1: `RUN_HILBERT = True` — get the first data point
- Session 2: `RUN_ALL = True`, paste Session 1 bpds into `PRIOR_RESULTS`

**Paper raster_v5 reference (historical, different N):**
`path_only` 2-spp raster: **6.43 ± 0.18 bpd** (3 seeds, 30 ep)


In [ ]:
import subprocess, sys
for _pkg in ["hilbertcurve", "datasets", "transformers", "open_clip_torch"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])
print("packages ready")


In [ ]:
import os, math, copy, json, random, statistics, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, List, Optional
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    from hilbertcurve.hilbertcurve import HilbertCurve as _HCLib
    _HAVE_HC = True
except ImportError:
    _HAVE_HC = False

LN2 = math.log(2)
torch.backends.cudnn.benchmark = True
print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()} | hilbertcurve lib: {_HAVE_HC}")


In [ ]:
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.expanduser("~/tmp/crt_traversal")
os.makedirs(WORK_DIR, exist_ok=True)

HF_TOKEN = None
for _src in [
    lambda: open("/kaggle/input/hf-token/token.txt").read().strip(),
    lambda: __import__("kaggle_secrets").UserSecretsClient().get_secret("HF_TOKEN"),
    lambda: os.environ["HF_TOKEN"],
]:
    try: HF_TOKEN = _src(); break
    except: pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"WORK_DIR = {WORK_DIR}")
print(f"device   = {DEVICE}")
print(f"HF_TOKEN = {'set' if HF_TOKEN else 'NOT SET (will try without)'}")


In [ ]:
# =========================================================================
# Traversal generators
# Each returns (path_np, beam_on_np):
#   path_np   : float32 [N, 2]  (x=col, y=row) in pixel-centre coords
#               e.g. pixel (col=0, row=0) -> (0.5, 0.5)
#   beam_on_np: bool    [N]     True = active, False = flyback (raster only)
# =========================================================================

def _d2xy(n, d):
    """Hilbert position d -> (col, row) for n x n grid (n = power of 2)."""
    x = y = 0
    s = 1
    while s < n:
        rx = 1 if (d & 2) else 0
        ry = 1 if ((d & 1) ^ rx) else 0
        if ry == 0:
            if rx == 1:
                x, y = s - 1 - x, s - 1 - y
            x, y = y, x
        x += s * rx
        y += s * ry
        d >>= 2
        s <<= 1
    return x, y

def hilbert_path(H, W):
    """Hilbert-curve traversal. N = H*W, all steps active (no flyback)."""
    if _HAVE_HC:
        n_bits = math.ceil(math.log2(max(H, W)))
        hc     = _HCLib(n_bits, 2)
        n_side = 2 ** n_bits
        coords = []
        for d in range(n_side * n_side):
            pt = hc.point_from_distance(d)
            col, row = int(pt[0]), int(pt[1])
            if col < W and row < H:
                coords.append((col + 0.5, row + 0.5))
    else:
        n = 1
        while n < max(H, W): n <<= 1
        coords = []
        for d in range(n * n):
            x, y = _d2xy(n, d)
            if x < W and y < H:
                coords.append((x + 0.5, y + 0.5))
    path = np.array(coords, dtype=np.float32)
    return path, np.ones(len(path), dtype=bool)

def raster_path(H, W, spp=1, flyback_frac=0.0):
    """
    Raster scan.
    spp=1, flyback_frac=0 -> clean 1:1 raster, N=H*W, all active.
    spp=2, flyback_frac=0.08 -> V5-style, N=2208 for 32x32.
    """
    active_per_row  = int(round(W * spp))
    flyback_per_row = max(0, int(round(active_per_row * flyback_frac / (1.0 - flyback_frac))))
    coords, mask = [], []
    for row in range(H):
        y = row + 0.5
        for i in range(active_per_row):
            coords.append(((i + 0.5) / spp, y)); mask.append(True)
        for j in range(flyback_per_row):
            frac = (j + 1) / (flyback_per_row + 1)
            coords.append((W * (1.0 - frac), y)); mask.append(False)
    return np.array(coords, dtype=np.float32), np.array(mask, dtype=bool)

def diagonal_path(H, W):
    """Anti-diagonal zigzag. N=H*W, no flyback. Preserves diagonal locality."""
    pixels = []
    for d in range(H + W - 1):
        if d % 2 == 0:
            for r in range(min(d, H-1), max(-1, d - W), -1):
                pixels.append((d - r + 0.5, r + 0.5))
        else:
            for c in range(min(d, W-1), max(-1, d - H), -1):
                pixels.append((c + 0.5, d - c + 0.5))
    return np.array(pixels, dtype=np.float32), np.ones(len(pixels), dtype=bool)

def spiral_path(H, W):
    """Inward clockwise spiral. N=H*W, no flyback."""
    visited = np.zeros((H, W), dtype=bool)
    dirs    = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # right, down, left, up (row_d, col_d)
    d_idx   = 0; r = c = 0; pixels = []
    for _ in range(H * W):
        pixels.append((c + 0.5, r + 0.5))
        visited[r, c] = True
        dr, dc = dirs[d_idx]; nr, nc = r + dr, c + dc
        if 0 <= nr < H and 0 <= nc < W and not visited[nr, nc]:
            r, c = nr, nc
        else:
            d_idx = (d_idx + 1) % 4
            r += dirs[d_idx][0]; c += dirs[d_idx][1]
    return np.array(pixels, dtype=np.float32), np.ones(len(pixels), dtype=bool)

TRAVERSALS = {
    "raster_v5"  : lambda H, W: raster_path(H, W, spp=2, flyback_frac=0.08),
    "raster_1spp": lambda H, W: raster_path(H, W, spp=1, flyback_frac=0.0),
    "hilbert"    : hilbert_path,
    "diagonal"   : diagonal_path,
    "spiral"     : spiral_path,
}
print("Traversal generators ready.")


In [ ]:
# =========================================================================
# Locality metric: mean sequence distance between spatially 4-adjacent
# pixel pairs.  Lower = better locality (transformer attends shorter range).
# =========================================================================

def locality_metric(path_np, H, W, beam_on_np=None):
    """Return statistics of sequence distances between spatially 4-adjacent pairs."""
    rows = np.clip(np.round(path_np[:, 1] - 0.5).astype(int), 0, H - 1)
    cols = np.clip(np.round(path_np[:, 0] - 0.5).astype(int), 0, W - 1)
    rc2idx = {}
    for i, (r, c) in enumerate(zip(rows, cols)):
        if beam_on_np is None or beam_on_np[i]:
            rc2idx.setdefault((r, c), i)
    dists = []
    for r in range(H):
        for c in range(W):
            if (r, c) not in rc2idx: continue
            idx = rc2idx[(r, c)]
            for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
                nb = (r+dr, c+dc)
                if nb in rc2idx:
                    dists.append(abs(idx - rc2idx[nb]))
    d = np.array(dists)
    return {
        "mean": float(d.mean()), "max": int(d.max()),
        "p25": float(np.percentile(d, 25)),
        "p50": float(np.percentile(d, 50)),
        "p75": float(np.percentile(d, 75)),
        "p95": float(np.percentile(d, 95)),
        "n_pairs": len(dists),
    }

# Always run (< 5 s) -------------------------------------------------------
H = W = 32
_loc_cache = {}  # reused in results cell
for name, fn in TRAVERSALS.items():
    path, beam_on = fn(H, W)
    _loc_cache[name] = (locality_metric(path, H, W, beam_on), len(path))

# Primary reference = raster_1spp (same N as all alternatives)
ref_mean = _loc_cache["raster_1spp"][0]["mean"]

print(f"{'Traversal':<22} {'N':>6}  {'mean':>6}  {'p50':>5}  {'p75':>5}  {'p95':>6}  {'max':>6}  {'vs raster_1spp':>15}")
print("-" * 85)
for name in ["raster_v5", "raster_1spp", "hilbert", "diagonal", "spiral"]:
    m, n = _loc_cache[name]
    ratio = f"{m['mean']/ref_mean:.2f}x"
    marker = " ← ref" if name == "raster_1spp" else ""
    print(f"{name:<22} {n:>6}  {m['mean']:>6.1f}  {m['p50']:>5.1f}  {m['p75']:>5.1f}  {m['p95']:>6.1f}  {m['max']:>6}  {ratio:>15}{marker}")
print()
print("Note: raster_1spp is the fair baseline (N=1024, no flyback, same as all alternatives).")
print("      raster_v5 is a historical reference only (N=2208, different token budget).")


In [ ]:
# ── Set to True to enable expensive training sections ──────────────────────
RUN_LOCALITY = True   # locality table above always runs; this flag is informational
RUN_HILBERT  = False  # ~3 h T4  |  Hilbert, 3 seeds x 30 epochs
RUN_ALL      = False  # ~12 h T4 |  all 4 traversals x 3 seeds x 30 epochs

# ── Paper raster_v5 baseline (already published, no need to re-run) ────────
PAPER_RASTER_BPD_MEAN = 6.432
PAPER_RASTER_BPD_STD  = 0.183

# ── Prior session results (paste numbers here to avoid re-training) ─────────
# Leave empty ({}) if starting fresh. Copy bpd_pixel values from previous run.
# Example after Run 1 (hilbert):
#   PRIOR_RESULTS = {"hilbert": [8.5079, 8.5015, 8.4650]}
PRIOR_RESULTS: dict = {}

print("Flags set. Paper raster baseline: "
      f"{PAPER_RASTER_BPD_MEAN} +/- {PAPER_RASTER_BPD_STD} bpd")
if PRIOR_RESULTS:
    print(f"Prior results loaded for: {list(PRIOR_RESULTS.keys())}")


In [ ]:
@dataclass
class Config:
    # image
    image_size:       int   = 32
    channels:         int   = 3
    # traversal (set per run)
    traversal:        str   = "hilbert"
    spp:              int   = 1
    flyback_frac:     float = 0.0
    # model
    d_model:          int   = 256
    n_heads:          int   = 4
    n_layers:         int   = 6
    ffn_mult:         int   = 4
    dropout:          float = 0.1
    n_mixtures:       int   = 5
    # renderer
    beam_sigma:       float = 0.75
    # positional encoding — always path-only (paper best config)
    use_path_pos_enc: bool  = True
    use_seq_pos_enc:  bool  = False
    # CLIP
    clip_model_name:  str   = "ViT-B/32"
    clip_dim:         int   = 512
    # training
    epochs:           int   = 30
    batch_size:       int   = 32
    lr:               float = 3e-4
    warmup_steps:     int   = 500
    grad_clip:        float = 1.0
    ss_prob_max:      float = 0.25
    # misc
    seed:             int   = 0
    device:           str   = DEVICE
    eval_every:       int   = 5
    ckpt_dir:         str   = f"{WORK_DIR}/ckpt"

BASE_CFG = Config()
print(BASE_CFG)


In [ ]:
FLOWER_NAMES = [
    "pink primrose","hard-leaved pocket orchid","canterbury bells","sweet pea",
    "english marigold","tiger lily","moon orchid","bird of paradise","monkshood",
    "globe thistle","snapdragon","colt's foot","king protea","spear thistle",
    "yellow iris","globe-flower","purple coneflower","peruvian lily",
    "balloon flower","giant white arum lily","fire lily","pincushion flower",
    "fritillary","red ginger","grape hyacinth","corn poppy",
    "prince of wales feathers","stemless gentian","artichoke","sweet william",
    "carnation","garden phlox","love in the mist","mexican aster",
    "alpine sea holly","ruby-lipped cattleya","cape flower","great masterwort",
    "siam tulip","lenten rose","barbeton daisy","daffodil","sword lily",
    "poinsettia","bolero deep blue","wallflower","marigold","buttercup",
    "oxeye daisy","common dandelion","petunia","wild pansy","primula",
    "sunflower","pelargonium","bishop of llandaff","gaura","geranium",
    "orange dahlia","pink-yellow dahlia","cautleya spicata","japanese anemone",
    "black-eyed susan","silverbush","californian poppy","osteospermum",
    "spring crocus","bearded iris","windflower","tree poppy","gazania",
    "azalea","water lily","rose","thorn apple","morning glory",
    "passion flower","lotus","toad lily","anthurium","frangipani",
    "clematis","hibiscus","columbine","desert-rose","tree mallow",
    "magnolia","cyclamen","watercress","canna lily","hippeastrum",
    "bee balm","ball moss","foxglove","bougainvillea","camellia",
    "mallow","mexican petunia","bromelia","blanket flower",
    "trumpet creeper","blackberry lily",
]

from torchvision import transforms
import open_clip
from datasets import load_dataset

class FlowerSignalDataset(Dataset):
    def __init__(self, hf_dataset, path_np, beam_on_np,
                 image_size, class_names, clip_model, clip_proc, device):
        self.data       = hf_dataset
        self.path_t     = torch.from_numpy(path_np.astype(np.float32))     # [N, 2]
        self.beam_on_t  = torch.from_numpy(beam_on_np.astype(np.float32))  # [N]
        self.image_size = image_size
        self.names      = class_names
        self.clip_model = clip_model
        self.clip_proc  = clip_proc
        self.device     = device
        self._cache     = {}
        self.tfm        = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])
        # Pre-compute ALL class embeddings now (main process, before any fork).
        # Workers receive a copy of self._cache and never touch the CLIP model,
        # avoiding the "Cannot re-initialize CUDA in forked subprocess" error.
        all_labels = sorted(set(item.get("label", 0) for item in hf_dataset))
        with torch.no_grad():
            for lbl in all_labels:
                name = self.names[lbl] if lbl < len(self.names) else f"flower {lbl}"
                toks = open_clip.tokenize([f"a photo of a {name}"])
                e    = clip_model.encode_text(toks.to(device))
                self._cache[lbl] = F.normalize(e, dim=-1).squeeze(0).cpu()

    def _embed(self, label):
        return self._cache[label]   # always hits cache (populated in __init__)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item  = self.data[idx]
        img   = self.tfm(item["image"].convert("RGB"))   # [3, H, W] in [0,1]
        label = item.get("label", 0)
        H = W = self.image_size
        # Normalize path to [-1, 1] (pixel-centre convention, align_corners=False)
        x_n = self.path_t[:, 0] * 2.0 / W - 1.0
        y_n = self.path_t[:, 1] * 2.0 / H - 1.0
        grid = torch.stack([x_n, y_n], dim=-1).unsqueeze(0).unsqueeze(0)  # [1,1,N,2]
        sig  = F.grid_sample(img.unsqueeze(0), grid,
                             mode="bilinear", align_corners=False)         # [1,3,1,N]
        sig  = sig.squeeze(0).squeeze(1).T                                  # [N, 3]
        sig  = sig * self.beam_on_t.unsqueeze(-1)
        return sig, self._embed(label), img


In [ ]:
print("Loading CLIP …")
_clip_model, _, _clip_proc = open_clip.create_model_and_transforms(
    BASE_CFG.clip_model_name, pretrained="openai")
_clip_model = _clip_model.to(DEVICE).eval()
for p in _clip_model.parameters(): p.requires_grad = False
print(f"  CLIP {BASE_CFG.clip_model_name}: "
      f"{sum(p.numel() for p in _clip_model.parameters())/1e6:.1f}M params (frozen)")

print("Loading Oxford Flowers 102 …")
_hf = load_dataset("nelorth/oxford-flowers",
                   token=HF_TOKEN if HF_TOKEN else None)
hf_train = _hf["train"]
hf_test  = _hf["test"]
print(f"  train={len(hf_train)}, test={len(hf_test)}")


In [ ]:
class DMoLHead(nn.Module):
    """Discretised Mixture of Logistics — identical to V5 paper."""
    def __init__(self, d_model, n_mix=5, C=3):
        super().__init__()
        self.K = n_mix; self.C = C
        self.proj = nn.Linear(d_model, n_mix * (1 + C * 2))

    def _unpack(self, x):
        B, N, _ = x.shape; K, C = self.K, self.C
        out   = self.proj(x)
        log_w = out[..., :K]
        loc   = out[..., K:K+K*C].view(B, N, K, C)
        log_s = out[..., K+K*C:].view(B, N, K, C).clamp(-7, 7)
        log_w = log_w - torch.logsumexp(log_w, -1, keepdim=True)
        return log_w, loc, log_s

    def nll(self, x, targets):
        """targets [B,N,C] in [0,1]. Returns nll [B,N]."""
        B, N, _ = x.shape
        log_w, loc, log_s = self._unpack(x)
        t   = (targets * 255.0).unsqueeze(-2).expand(B, N, self.K, self.C)
        c   = (t - loc) * (-log_s).exp()
        inv_s = (-log_s).exp()
        p   = torch.sigmoid(c + 0.5 * inv_s)
        m   = torch.sigmoid(c - 0.5 * inv_s)
        lp  = (p - m).clamp(1e-12).log()
        lp  = torch.where(t <   0.5, p.clamp(1e-12).log(), lp)
        lp  = torch.where(t > 254.5, (1 - m).clamp(1e-12).log(), lp)
        return -(log_w + lp.sum(-1)).logsumexp(-1)   # [B,N]

    @torch.no_grad()
    def sample(self, x, temp=1.0):
        B, N, _ = x.shape
        log_w, loc, log_s = self._unpack(x)
        k  = torch.multinomial((log_w / temp).softmax(-1).view(-1, self.K), 1).view(B, N)
        ke = k.unsqueeze(-1).unsqueeze(-1).expand(B, N, 1, self.C)
        mu = loc.gather(2, ke).squeeze(2)
        ls = log_s.gather(2, ke).squeeze(2)
        u  = torch.empty_like(mu).uniform_(1e-5, 1 - 1e-5)
        s  = mu + ls.exp() * (u.log() - (1 - u).log()) * temp
        return (s.clamp(0, 255) / 255.0)


In [ ]:
class CRTRenderer(nn.Module):
    """Gaussian-beam differentiable renderer (V5, traversal-agnostic)."""
    def __init__(self, path_np, beam_on_np, image_size, sigma=0.75):
        super().__init__()
        H = W = image_size
        path = torch.from_numpy(path_np.astype(np.float32))    # [N,2]
        mask = torch.from_numpy(beam_on_np.astype(np.float32)) # [N]
        ys   = torch.arange(H, dtype=torch.float32) + 0.5      # [H]
        xs   = torch.arange(W, dtype=torch.float32) + 0.5      # [W]
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")         # [H,W]
        dy2  = (gy.unsqueeze(0) - path[:, 1].view(-1, 1, 1)) ** 2
        dx2  = (gx.unsqueeze(0) - path[:, 0].view(-1, 1, 1)) ** 2
        kern = torch.exp(-(dx2 + dy2) / (2 * sigma ** 2))      # [N,H,W]
        n_active = max(1, int(beam_on_np.sum()))
        norm     = n_active * 2 * math.pi * sigma ** 2
        self.register_buffer("kern", kern / norm)
        self.register_buffer("mask", mask)
        self.image_size = image_size

    def forward(self, signal):
        """signal [B,N,C] -> image [B,C,H,W]"""
        m = signal * self.mask.unsqueeze(-1)
        return torch.einsum("bnc,nhw->bchw", m, self.kern)


In [ ]:
class SinPE1D(nn.Module):
    def __init__(self, d_model, max_len=8192):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d]

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SinPE2D(nn.Module):
    """Half channels encode x-coord (col), half encode y-coord (row)."""
    def __init__(self, d_model, image_size):
        super().__init__()
        assert d_model % 2 == 0
        self.d_half     = d_model // 2
        self.image_size = image_size
        div = torch.exp(torch.arange(0, self.d_half, 2) * (-math.log(10000.0) / self.d_half))
        self.register_buffer("div", div)

    def forward(self, x, path_np):
        N, D = x.size(1), self.d_half
        S    = float(self.image_size)
        px   = torch.from_numpy(path_np[:, 0]).float().to(x.device).unsqueeze(-1) / S  # [N,1]
        py   = torch.from_numpy(path_np[:, 1]).float().to(x.device).unsqueeze(-1) / S
        pe   = torch.zeros(1, N, x.size(-1), device=x.device)
        pe[0, :, 0:D:2]      = torch.sin(px * self.div)
        pe[0, :, 1:D:2]      = torch.cos(px * self.div)
        pe[0, :, D:D+D:2]    = torch.sin(py * self.div)
        pe[0, :, D+1:D+D:2]  = torch.cos(py * self.div)
        return x + pe

class FiLM(nn.Module):
    def __init__(self, d_model, cond_dim):
        super().__init__()
        self.to_scale_shift = nn.Linear(cond_dim, 2 * d_model)

    def forward(self, x, c):
        g, b = self.to_scale_shift(c).chunk(2, dim=-1)
        return x * (1 + g.unsqueeze(1)) + b.unsqueeze(1)


In [ ]:
class CausalBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_mult=4, dropout=0.1, cond_dim=512):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads,
                                          dropout=dropout, batch_first=True)
        self.film = FiLM(d_model, cond_dim)
        self.ff   = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * ffn_mult),
            nn.GELU(),
            nn.Linear(d_model * ffn_mult, d_model),
            nn.Dropout(dropout),
        )
        self.ln1  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, cond, attn_mask):
        h    = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=attn_mask, is_causal=True)
        x    = x + self.drop(h)
        x    = x + self.film(x, cond)
        return x + self.ff(x)

class SignalTransformer(nn.Module):
    def __init__(self, cfg: Config, path_np: np.ndarray, beam_on_np: np.ndarray):
        super().__init__()
        self.cfg     = cfg
        self.path_np = path_np   # kept as numpy; used in SinPE2D.forward()

        self.input_proj = nn.Linear(cfg.channels, cfg.d_model)
        self.cond_proj  = nn.Linear(cfg.clip_dim, cfg.d_model)
        self.seq_pe     = SinPE1D(cfg.d_model) if cfg.use_seq_pos_enc  else None
        self.path_pe    = SinPE2D(cfg.d_model, cfg.image_size) if cfg.use_path_pos_enc else None

        self.blocks = nn.ModuleList([
            CausalBlock(cfg.d_model, cfg.n_heads, cfg.ffn_mult,
                        cfg.dropout, cfg.d_model)
            for _ in range(cfg.n_layers)
        ])
        self.ln_out = nn.LayerNorm(cfg.d_model)
        self.head   = DMoLHead(cfg.d_model, cfg.n_mixtures, cfg.channels)

    @staticmethod
    def _causal_mask(N, device):
        return torch.triu(torch.full((N, N), float("-inf"), device=device), diagonal=1)

    def _encode(self, signal_in, clip_emb):
        B, N, _  = signal_in.shape
        x        = self.input_proj(signal_in)
        cond     = self.cond_proj(clip_emb)
        if self.seq_pe:  x = self.seq_pe(x)
        if self.path_pe: x = self.path_pe(x, self.path_np)
        mask     = self._causal_mask(N, signal_in.device)
        for blk in self.blocks:
            x = blk(x, cond, mask)
        return self.ln_out(x)

    def forward(self, signal, clip_emb, ss_prob=0.0):
        """signal [B,N,C], clip_emb [B,D]. Returns nll [B,N]."""
        B, N, C = signal.shape
        if ss_prob > 0.0 and self.training:
            with torch.no_grad():
                preds = self.head.sample(self._encode(signal, clip_emb)[:, :-1])
            mix   = torch.rand(B, N - 1, 1, device=signal.device) < ss_prob
            s_in  = torch.cat([signal[:, :1],
                                torch.where(mix, preds, signal[:, :-1])], dim=1)
        else:
            s_in = signal
        return self.head.nll(self._encode(s_in, clip_emb), signal)   # [B,N]


In [ ]:
def train(cfg, model, clip_model, train_ds, val_ds=None, ckpt_dir=None):
    if ckpt_dir: os.makedirs(ckpt_dir, exist_ok=True)
    loader  = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)
    opt     = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                                betas=(0.9, 0.95), weight_decay=1e-2)
    beam_on = torch.from_numpy(train_ds.beam_on_t.numpy().astype(bool)).to(cfg.device)
    n_steps = len(loader) * cfg.epochs

    def lr_fn(step):
        if step < cfg.warmup_steps: return step / max(1, cfg.warmup_steps)
        p = (step - cfg.warmup_steps) / max(1, n_steps - cfg.warmup_steps)
        return max(0.01, 0.5 * (1 + math.cos(math.pi * p)))

    sched  = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.device == "cuda"))
    best_val = float("inf"); step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total = 0.0; nb = 0
        ss_p  = cfg.ss_prob_max * min(1.0, epoch / cfg.epochs)
        for sig, emb, _ in loader:
            sig = sig.to(cfg.device); emb = emb.to(cfg.device)
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=(cfg.device == "cuda")):
                nll  = model(sig, emb, ss_prob=ss_p)
                loss = (nll * beam_on.float()).sum(-1).mean() / beam_on.float().sum()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            total += loss.item(); nb += 1; step += 1

        avg = total / nb
        if epoch % cfg.eval_every == 0 or epoch == cfg.epochs:
            vm = evaluate(model, clip_model, val_ds, cfg) if val_ds else {}
            print(f"epoch {epoch:>3}/{cfg.epochs}  avg_nll={avg:.4f}",
                  f"  val bpd={vm['bpd_pixel']:.4f}" if vm else "")
            if vm and vm["bpd_pixel"] < best_val and ckpt_dir:
                best_val = vm["bpd_pixel"]
                torch.save({"model": model.state_dict(), "epoch": epoch,
                            "bpd": best_val},
                           os.path.join(ckpt_dir, "best.pt"))
        else:
            print(f"epoch {epoch:>3}/{cfg.epochs}  avg_nll={avg:.4f}")

    if ckpt_dir:
        torch.save({"model": model.state_dict(), "epoch": cfg.epochs},
                   os.path.join(ckpt_dir, f"epoch_{cfg.epochs}.pt"))
    return evaluate(model, clip_model, val_ds, cfg) if val_ds else {}

@torch.no_grad()
def evaluate(model, clip_model, dataset, cfg):
    model.eval()
    loader  = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)
    beam_on = torch.from_numpy(dataset.beam_on_t.numpy().astype(bool)).to(cfg.device)
    total = 0.0; n = 0
    for sig, emb, _ in loader:
        sig = sig.to(cfg.device); emb = emb.to(cfg.device)
        nll = model(sig, emb, ss_prob=0.0)
        total += (nll * beam_on.float()).sum(-1).sum().item()
        n     += sig.size(0)
    mean_nll   = total / max(1, n)
    image_dims = cfg.image_size ** 2 * cfg.channels
    model.train()
    return {"bpd_pixel": mean_nll / LN2 / image_dims,
            "nll_per_image": mean_nll, "n_images": n}


In [ ]:
def sanity_check(traversal="hilbert"):
    path_np, beam_on_np = TRAVERSALS[traversal](BASE_CFG.image_size, BASE_CFG.image_size)
    cfg  = copy.deepcopy(BASE_CFG); cfg.traversal = traversal
    m    = SignalTransformer(cfg, path_np, beam_on_np).to(DEVICE)
    n_p  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    B    = 2; N = len(path_np)
    sig  = torch.rand(B, N, 3, device=DEVICE)
    emb  = torch.randn(B, 512, device=DEVICE)
    nll  = m(sig, emb)
    assert nll.shape == (B, N), f"Got {nll.shape}, expected [{B},{N}]"
    print(f"traversal   = {traversal}  (N={N})")
    print(f"params      = {n_p:,}  ({n_p/1e6:.2f}M)")
    print(f"nll shape   = {nll.shape}   OK")
    print("Sanity check PASSED")
    return n_p

_ = sanity_check("hilbert")


## Main Experiment: Controlled Traversal Comparison

All models are **identical** — only the pixel-visitation order changes.
The **fair comparison** is any alternative vs `raster_1spp` (same N = 1024).

| | raster_1spp (primary baseline) | hilbert / diagonal / spiral |
|---|---|---|
| N (32×32) | 1024 | 1024 |
| Path PE | yes | yes |
| Seq PE | no | no |
| Model params | ~5.7M | ~5.7M |
| Epochs | 30 | 30 |
| Seeds | 3 | 3 |

`raster_v5` (N = 2208) is shown in the summary table as a historical
reference from the V5 paper but is **not used as a comparator** because
it has a different token budget.

The paper question: does the locality metric (from the table above)
**predict** bpd ranking across traversals?


In [ ]:
def run_traversal(traversal_name, seeds=(0, 1, 2), n_epochs=30):
    """Train traversal_name for each seed. Returns list of metric dicts."""
    results = []
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        path_np, beam_on_np = TRAVERSALS[traversal_name](
            BASE_CFG.image_size, BASE_CFG.image_size)
        cfg           = copy.deepcopy(BASE_CFG)
        cfg.traversal = traversal_name
        cfg.seed      = seed
        cfg.epochs    = n_epochs
        cfg.ckpt_dir  = f"{WORK_DIR}/ckpt_{traversal_name}_s{seed}"
        model         = SignalTransformer(cfg, path_np, beam_on_np).to(DEVICE)
        train_ds      = FlowerSignalDataset(hf_train, path_np, beam_on_np,
                            cfg.image_size, FLOWER_NAMES,
                            _clip_model, _clip_proc, DEVICE)
        test_ds       = FlowerSignalDataset(hf_test, path_np, beam_on_np,
                            cfg.image_size, FLOWER_NAMES,
                            _clip_model, _clip_proc, DEVICE)
        print(f"\n=== {traversal_name}  seed={seed}  N={len(path_np)} ===")
        m = train(cfg, model, _clip_model, train_ds, val_ds=test_ds,
                  ckpt_dir=cfg.ckpt_dir)
        print(f"  bpd_pixel = {m['bpd_pixel']:.4f}")
        results.append(m)
    return results

# Seed all_results from any prior-session numbers (bpd lists -> metric dicts)
all_results: Dict[str, list] = {}
for tname, bpds in PRIOR_RESULTS.items():
    all_results[tname] = [{"bpd_pixel": b, "nll_per_image": b * LN2 * BASE_CFG.image_size**2 * BASE_CFG.channels}
                          for b in bpds]
    print(f"  Loaded prior {tname}: {[round(b,4) for b in bpds]}")

if RUN_ALL:
    for tname in ["hilbert", "diagonal", "spiral", "raster_1spp"]:
        if tname in all_results:
            print(f"  Skipping {tname} — already in PRIOR_RESULTS.")
            continue
        all_results[tname] = run_traversal(tname)
elif RUN_HILBERT:
    if "hilbert" in all_results:
        print("hilbert already in PRIOR_RESULTS — skipping training.")
    else:
        all_results["hilbert"] = run_traversal("hilbert")
else:
    print("No RUN flag is True.")
    print("Set  RUN_HILBERT = True  to train the Hilbert model (~3 h on T4).")
    print("Set  RUN_ALL     = True  for the full comparison (~12 h on T4).")


In [ ]:
# Save any new results
out_path = f"{WORK_DIR}/traversal_results.json"
with open(out_path, "w") as f:
    json.dump(
        {k: [{"bpd_pixel": r["bpd_pixel"], "nll_per_image": r["nll_per_image"]}
             for r in v]
         for k, v in all_results.items()},
        f, indent=2)
print(f"Results saved -> {out_path}")

# ── Summary table ──────────────────────────────────────────────────────────
# Primary baseline = raster_1spp (same N).  raster_v5 shown as reference.
summary = {"raster_v5 (ref, N=2208)": (PAPER_RASTER_BPD_MEAN, PAPER_RASTER_BPD_STD)}
for tname, res_list in all_results.items():
    bpds = [r["bpd_pixel"] for r in res_list]
    mu   = statistics.mean(bpds)
    sd   = statistics.stdev(bpds) if len(bpds) > 1 else 0.0
    summary[tname] = (mu, sd)

# Use raster_1spp as primary ref if available; fall back to paper number
if "raster_1spp" in all_results:
    bpds_r = [r["bpd_pixel"] for r in all_results["raster_1spp"]]
    PRIMARY_REF       = statistics.mean(bpds_r)
    PRIMARY_REF_LABEL = "raster_1spp"
else:
    PRIMARY_REF       = PAPER_RASTER_BPD_MEAN
    PRIMARY_REF_LABEL = "raster_v5 (paper fallback)"

print("\n" + "=" * 72)
print("TRAVERSAL COMPARISON — Oxford Flowers 32x32, 30 ep, 3 seeds")
print(f"Primary baseline: {PRIMARY_REF_LABEL} = {PRIMARY_REF:.4f} bpd")
print("=" * 72)
print(f"{'Traversal':<28}  {'bpd mean':>10}  {'bpd std':>8}  {'vs primary ref':>15}")
print("-" * 72)
for name, (mu, sd) in sorted(summary.items(), key=lambda x: x[1][0]):
    if "ref" in name:
        delta = "  (hist. ref)"
    elif name == PRIMARY_REF_LABEL:
        delta = "  ← baseline"
    else:
        delta = f"{mu - PRIMARY_REF:+.3f}"
    print(f"{name:<28}  {mu:>10.4f}  {sd:>8.4f}  {delta:>15}")

# ── Failure gate ───────────────────────────────────────────────────────────
IMPROVEMENT_THRESHOLD = 0.10   # bpd; must beat baseline by this much to claim win
print()
any_improvement = False
for tname in ["hilbert", "diagonal", "spiral"]:
    if tname not in all_results:
        continue
    bpds = [r["bpd_pixel"] for r in all_results[tname]]
    mu   = statistics.mean(bpds)
    gain = PRIMARY_REF - mu
    if gain > IMPROVEMENT_THRESHOLD:
        print(f"  RESULT: {tname} IMPROVED over {PRIMARY_REF_LABEL} by {gain:.4f} bpd  ✓")
        any_improvement = True
    else:
        print(f"  RESULT: {tname} did NOT beat {PRIMARY_REF_LABEL} "
              f"(delta = {gain:+.4f} bpd, threshold = {IMPROVEMENT_THRESHOLD}).  ✗")

if not any_improvement and len(all_results) > 0:
    print()
    print("  CONCLUSION: No traversal improves over raster_1spp at equal N.")
    print("  Frame as negative result: traversal geometry alone does not predict")
    print("  likelihood at this scale. Do NOT generate samples or draft paper")
    print("  claims from these runs without additional evidence.")
elif any_improvement:
    print()
    print("  Set RUN_SAMPLES=True to generate samples from the best traversal.")


## Optional: Generate Samples from Best Checkpoint

Run the cell below after training to visualise Hilbert-curve generated images
and compare with raster (V5) samples.


In [ ]:
RUN_SAMPLES = False   # set True after training completes

PROMPTS = [
    "a photo of a red rose",
    "a photo of a yellow sunflower",
    "a photo of a purple iris",
    "a photo of a white lily",
    "a photo of a pink daisy",
    "a photo of an orange marigold",
]

if RUN_SAMPLES and "hilbert" in all_results:
    cfg_s = copy.deepcopy(BASE_CFG); cfg_s.traversal = "hilbert"
    path_np, beam_on_np = hilbert_path(cfg_s.image_size, cfg_s.image_size)
    model_s = SignalTransformer(cfg_s, path_np, beam_on_np).to(DEVICE)
    ckpt_path = f"{WORK_DIR}/ckpt_hilbert_s0/best.pt"
    ckpt      = torch.load(ckpt_path, map_location=DEVICE)
    model_s.load_state_dict(ckpt["model"])
    model_s.eval()
    renderer  = CRTRenderer(path_np, beam_on_np,
                            cfg_s.image_size, cfg_s.beam_sigma).to(DEVICE)

    fig, axes = plt.subplots(1, len(PROMPTS), figsize=(3 * len(PROMPTS), 3))
    for ax, prompt in zip(axes, PROMPTS):
        toks = open_clip.tokenize([prompt])
        with torch.no_grad():
            emb = _clip_model.encode_text(toks.to(DEVICE))
            emb = F.normalize(emb, dim=-1)
            N   = len(path_np)
            sig = torch.zeros(1, N, 3, device=DEVICE)
            for i in range(N):
                if beam_on_np[i]:
                    nll_out = model_s._encode(sig, emb)
                    sig[0, i] = model_s.head.sample(nll_out[:, i:i+1]).squeeze(1)
            img = renderer(sig).squeeze(0).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
        ax.imshow(img); ax.set_title(prompt.replace("a photo of a ", ""), fontsize=8)
        ax.axis("off")
    plt.suptitle("Hilbert traversal — generated samples (seed 0, best checkpoint)")
    plt.tight_layout()
    plt.savefig(f"{WORK_DIR}/hilbert_samples.png", dpi=120)
    plt.show()
    print(f"Saved -> {WORK_DIR}/hilbert_samples.png")
else:
    print("Set RUN_SAMPLES=True after RUN_HILBERT training completes.")
